In [0]:
# ============================================================
# Gold Layer - Business Aggregations
# ============================================================

from pyspark.sql.functions import *

# Read Silver Table
orders = spark.table("workspace.pei_assignment.order_enriched")

# ============================================================
# Total Sales & Profit by Category
# ============================================================

category_summary = (
    orders
    .groupBy("category")
    .agg(
        round(sum("price"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit"),
        sum("quantity").alias("total_quantity")
    )
    .orderBy(desc("total_profit"))
)

category_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.pei_assignment.gold_category_summary")

# ============================================================
# Customer Sales Summary
# ============================================================

customer_summary = (
    orders
    .groupBy(
        "customer_id",
        "customer_name",
        "segment"
    )
    .agg(
        round(sum("price"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy(desc("total_sales"))
)

customer_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.pei_assignment.gold_customer_summary")

# ============================================================
# State Sales Summary
# ============================================================

state_summary = (
    orders
    .groupBy("customer_state")
    .agg(
        round(sum("price"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit")
    )
    .orderBy(desc("total_sales"))
)

state_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.pei_assignment.gold_state_summary")

# ============================================================
# Yearly Sales Trend
# ============================================================

yearly_summary = (
    orders
    .withColumn(
        "order_year",
        year(to_date(col("order_date"), "d/M/yyyy"))
    )
    .groupBy("order_year")
    .agg(
        round(sum("price"), 2).alias("total_sales"),
        round(sum("profit"), 2).alias("total_profit")
    )
    .orderBy("order_year")
)

yearly_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.pei_assignment.gold_yearly_summary")

# ============================================================
# Verification
# ============================================================

print("=" * 60)
print("Gold Layer Verification")
print("=" * 60)

spark.sql("""
SHOW TABLES IN workspace.pei_assignment
""").show(truncate=False)

display(category_summary)
display(customer_summary)
display(state_summary)
display(yearly_summary)

print("Gold Layer Created Successfully")

Gold Layer Verification
+--------------+---------------------+-----------+
|database      |tableName            |isTemporary|
+--------------+---------------------+-----------+
|pei_assignment|customer_enriched    |false      |
|pei_assignment|customer_raw         |false      |
|pei_assignment|gold_category_summary|false      |
|pei_assignment|gold_customer_summary|false      |
|pei_assignment|gold_state_summary   |false      |
|pei_assignment|gold_yearly_summary  |false      |
|pei_assignment|order_enriched       |false      |
|pei_assignment|orders_raw           |false      |
|pei_assignment|product_enriched     |false      |
|pei_assignment|products_raw         |false      |
+--------------+---------------------+-----------+



category,total_sales,total_profit,total_quantity
Technology,900817.46,142942.26,6937
Office Supplies,915419.47,125848.36,22215
Furniture,868666.88,7627.18,7961
null,8338.8,1999.23,760


customer_id,customer_name,segment,total_sales,total_profit,total_orders
DB-13555,Dorothy Badders,Corporate,97244.02,109.63,7
DP-13105,Dave Poirier,Corporate,49733.92,562.95,8
JE-15745,Joel Eaton,Consumer,46044.06,222.24,13
SC-20770,Stewart Carmichael,Corporate,42773.84,-671.38,7
LH-17155,Logan Haushalter,Consumer,35205.72,316.53,9
FH-14365,Fred Hopkins,Corporate,33778.58,2051.01,8
SM-20320,Sean Miller,Home Office,25043.07,-1980.98,5
SC-20020,Sam Craven,Consumer,22476.89,-317.51,5
PS-18970,Paul Stevenson,Home Office,22377.03,198.53,8
JF-15190,Jamie Frazer,Consumer,21646.83,575.78,7


customer_state,total_sales,total_profit
California,466274.82,62805.95
New York,327632.83,49041.91
Texas,269602.47,35662.45
North Carolina,173990.47,4942.67
Pennsylvania,144548.9,5618.58
Ohio,129565.8,-16479.24
Washington,124650.42,13843.28
Michigan,92074.44,12250.02
Tennessee,87676.61,1655.08
Florida,80223.97,5146.18


order_year,total_sales,total_profit
2014,510200.55,39185.75
2015,542453.97,63073.09
2016,762385.09,65073.23
2017,878203.0,111084.96


Gold Layer Created Successfully
